In [ ]:
# 가상환경 위치 확인
# !pip --version

pip 26.2.1 from E:\hanwh2609\rag_one\.venv\Lib\site-packages\pip (python 3.12)



In [2]:
from dotenv import load_dotenv

# .env파일에 설정된 보안정보를 읽기.
load_dotenv()

True

### RAG 심플 동작

In [ ]:
# !pip install -U langchain langchain-openai

In [4]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [6]:
# 문서 정의
documents = [
    Document(
        page_content="연차 휴가는 그룹웨어에서 신청합니다.",
        metadata={"source": "인사규정", "page": 3},
    ),
    Document(
        page_content="비밀번호는 보안 포털에서 변경합니다.",
        metadata={"source": "보안지침", "page": 7},
    ),
]

# documents벡터값 저장
store = InMemoryVectorStore.from_documents(
    documents,
    OpenAIEmbeddings(model="text-embedding-3-small"),
)

# 검색기-유사검색 상위1개
retriever = store.as_retriever(search_kwargs={"k": 1})

# 요청 프롬프트 - 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    "문서만 사용해 답하세요.\n문서: {context}\n질문: {question}"
)

# 모델 정의
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# LECL - 체인셋팅
rag_chain = prompt | model | StrOutputParser()

In [7]:
print(retriever)

tags=['InMemoryVectorStore', 'OpenAIEmbeddings'] vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x0000016299DCFCB0> search_kwargs={'k': 1}


In [8]:
# 사용자 질문
question = "연차 휴가는 어디에서 신청하나요?"

# Runnable Interface
found = retriever.invoke(question)


In [9]:
print(found)

[Document(id='1322d847-86bb-42b2-9fd6-9e3c462e64a7', metadata={'source': '인사규정', 'page': 3}, page_content='연차 휴가는 그룹웨어에서 신청합니다.')]


In [10]:
# 참고자료 생성
context = "\n\n".join(doc.page_content for doc in found)
print(context)

연차 휴가는 그룹웨어에서 신청합니다.


In [11]:

# LLM 답변
answer = rag_chain.invoke({"question": question, "context": context})


In [12]:
print("답변:", answer)
print("출처:", [doc.metadata for doc in found])

답변: 연차 휴가는 그룹웨어에서 신청합니다.
출처: [{'source': '인사규정', 'page': 3}]
